# Few-shot 예시 선택

예시가 없을 때, 관련 예시가 있을 때, 형식만 비슷한 관련 없는 예시가 있을 때 결과는 어떻게 달라질까요?

## 패키지 준비

첫 실행에서 수업 기준 버전을 설치합니다.

In [1]:
# 현재 커널에 수업 기준 패키지 버전을 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## API 키와 모델 준비

상위 폴더의 `.env`를 찾고 키의 존재만 확인합니다.

In [2]:
# find_dotenv는 실행 폴더부터 상위 폴더의 .env를 찾고 키 값은 출력하지 않습니다.
import json, os, time
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 루트의 .env에 OPENAI_API_KEY가 필요합니다.")
model = init_chat_model("openai:gpt-5.4-mini", max_tokens=220)

## 고정 과제

모든 절에서 같은 고객 문의와 확인 단어를 사용합니다.

In [3]:
# TASK는 모든 비교에서 바꾸지 않는 합성 문의이고 MENTIONS는 품질 점수가 아닙니다.
TASK = """고객 문의: 어제 받은 텀블러 뚜껑이 깨졌습니다. 주문번호는 A-1042입니다.
새 상품으로 바꾸고 싶습니다. 회사 규정상 교환 접수에는 파손 사진이 필요합니다.
답변에는 분류, 필요한 다음 행동, 고객에게 보낼 한 문장을 포함하세요."""
MENTIONS = ["교환", "A-1042", "사진"]

## 측정 함수

단어 포함 표시는 원문 정독을 돕는 표시이며 품질 점수가 아닙니다.

In [4]:
# 모델 응답의 원문, 사용량, 경과 시간을 한 레코드로 보존합니다.
def run(messages):
    started = time.perf_counter()
    answer = model.invoke(messages)
    return {"text": answer.text, "usage": answer.usage_metadata,
            "latency_s": round(time.perf_counter() - started, 3)}

def mention_flags(text):
    return {word: word in text for word in MENTIONS}

## 예시 없이 실행

출력 형식만 설명합니다.

In [5]:
# 입출력 예시가 없는 기준 응답을 먼저 저장합니다.
zero = run([("system", "분류/다음 행동/고객 문장 형식으로 답하세요."), ("user", TASK)])
print(json.dumps(zero, ensure_ascii=False, indent=2))

{
  "text": "분류: 교환/파손 접수\n\n다음 행동: 파손된 뚜껑 사진을 요청하고, 주문번호 A-1042로 교환 접수를 진행하세요.\n\n고객 문장: 교환 접수를 위해 깨진 뚜껑의 사진을 보내주시면 확인 후 새 상품으로 교환 도와드리겠습니다.",
  "usage": {
    "input_tokens": 102,
    "output_tokens": 81,
    "total_tokens": 183,
    "input_token_details": {
      "audio": 0,
      "cache_read": 0
    },
    "output_token_details": {
      "audio": 0,
      "reasoning": 0
    }
  },
  "latency_s": 1.143
}


## 관련 예시와 관련 없는 예시 비교

두 조건 모두 예시는 하나이며, 현재 질문과의 관련성만 다릅니다.

In [6]:
# 관련 예시와 관련 없는 예시는 한 개씩 두고 현재 질문은 고정합니다.
relevant_example = """예시 문의: 셔츠 단추가 떨어졌습니다. 주문번호 B-203입니다.
예시 답: 분류=교환 / 다음 행동=하자 사진과 주문번호 확인 / 고객 문장=사진을 첨부해 주세요."""
irrelevant_example = """예시 문의: 뉴스레터 글꼴을 바꾸고 싶습니다.
예시 답: 분류=설정 / 다음 행동=설정 메뉴 확인 / 고객 문장=글꼴 메뉴를 선택해 주세요."""
few = run([("system", "분류/다음 행동/고객 문장 형식으로 답하세요."),
           ("user", relevant_example + "\n\n" + TASK)])
off = run([("system", "분류/다음 행동/고객 문장 형식으로 답하세요."),
           ("user", irrelevant_example + "\n\n" + TASK)])

## 세 조건의 원문과 점수

예시가 정답을 보장하는지, 입력 토큰이 얼마나 늘었는지도 확인합니다.

In [7]:
# 예시 없음·관련 예시·관련 없는 예시의 raw 응답을 모두 출력합니다.
for name, result in {"zero_shot": zero, "relevant": few, "unrelated": off}.items():
    print("\n", name, json.dumps(mention_flags(result["text"]), ensure_ascii=False),
          result["usage"], result["latency_s"])
    print(result["text"])


 zero_shot {"교환": true, "A-1042": true, "사진": true} {'input_tokens': 102, 'output_tokens': 81, 'total_tokens': 183, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.143
분류: 교환/파손 접수

다음 행동: 파손된 뚜껑 사진을 요청하고, 주문번호 A-1042로 교환 접수를 진행하세요.

고객 문장: 교환 접수를 위해 깨진 뚜껑의 사진을 보내주시면 확인 후 새 상품으로 교환 도와드리겠습니다.

 relevant {"교환": true, "A-1042": false, "사진": true} {'input_tokens': 156, 'output_tokens': 40, 'total_tokens': 196, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 0.775
분류=교환 / 다음 행동=파손 사진과 주문번호 확인 / 고객 문장=파손된 뚜껑 사진을 첨부해 주세요.

 unrelated {"교환": true, "A-1042": true, "사진": true} {'input_tokens': 150, 'output_tokens': 64, 'total_tokens': 214, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 0.858
분류=교환/파손 접수 / 다음 행동=파손 사진 요청 및 교환 접수 진행 / 고객 문장=파손된 뚜껑 사진을 보내주시면 주문번호 A-1042로 새 상품 교환 접수를 도와드리겠습니다.
